# 03 - Climate Feature Engineering

**Architecture block 2b.** Raw climate -> scaling -> temporal aggregation ->
agricultural feature set.

The derived variables are the ones plant pathologists actually use, because
raw temperature alone does not drive infection.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent / "src"))

import cropforecast
from cropforecast.config import load_config, ensure_dirs, set_seed, Device
cfg = load_config(Path.cwd().parent / "configs" / "default.yaml")
ensure_dirs(cfg); set_seed(cfg.project.seed)
device = Device.auto(cfg.training.amp)
print("device:", device)

In [ ]:
import pandas as pd, numpy as np
from cropforecast.features.agromet import build_climate_features, derive_daily_features
raw = pd.read_csv(Path(cfg.paths.climate) / "HP-SML.csv", parse_dates=["date"])
feat = build_climate_features(raw, windows=tuple(cfg.climate.rolling_windows))
print(f"{raw.shape[1]} raw columns -> {feat.shape[1]} engineered columns")
[c for c in feat.columns if c not in raw.columns][:25]

### The derived quantities

- **VPD** - drying power of the air (Tetens equation)
- **Leaf wetness hours** - the single best predictor of fungal infection
- **Dew-point depression** - how close the air is to condensing
- **GDD** - crop development clock
- **wind_u / wind_v** - spore transport vector

In [ ]:
import matplotlib.pyplot as plt
sub = feat[feat.date.dt.year == 2023]
fig, ax = plt.subplots(4, 1, figsize=(13, 9), sharex=True)
ax[0].plot(sub.date, sub.temperature_2m_mean, color="#f43f5e"); ax[0].set_ylabel("Temp C")
ax[1].plot(sub.date, sub.relative_humidity_2m_mean, color="#38bdf8"); ax[1].set_ylabel("RH %")
ax[2].plot(sub.date, sub.leaf_wetness_hours, color="#22c55e"); ax[2].set_ylabel("Leaf wet h")
ax[3].plot(sub.date, sub.vpd_kpa, color="#fbbf24"); ax[3].set_ylabel("VPD kPa")
fig.suptitle("Shimla 2023 - monsoon is unmistakable"); plt.tight_layout(); plt.show()

## The agronomic knowledge base

Each pathogen has its own environmental response. Several are *opposites*.

In [ ]:
from cropforecast.physics.epidemiology import PROFILES, temperature_response
import numpy as np
t = np.linspace(0, 42, 300)
fig, ax = plt.subplots(figsize=(11, 5))
for name in ["Potato___Late_blight","Apple___Apple_scab","Squash___Powdery_mildew",
             "Tomato___Spider_mites Two-spotted_spider_mite","Tomato___Bacterial_spot"]:
    ax.plot(t, temperature_response(t, PROFILES[name]), label=name.split("___")[1].replace("_"," "), lw=2)
ax.set_xlabel("Temperature C"); ax.set_ylabel("relative development rate")
ax.set_title("Analytis beta model - cardinal temperatures per pathogen")
ax.legend(); ax.grid(alpha=.3); plt.show()

### Favourability on real weather

Wettest vs driest recorded day at Shimla.

In [ ]:
from cropforecast.physics.epidemiology import favourability
wet = feat.loc[[feat.leaf_wetness_hours.idxmax()]]
dry = feat.loc[[feat.vpd_kpa.idxmax()]]
classes = ["Apple___Apple_scab","Potato___Late_blight","Squash___Powdery_mildew",
           "Tomato___Spider_mites Two-spotted_spider_mite","Apple___healthy"]
pd.DataFrame({
    "wettest day": [favourability(c, wet)[0] for c in classes],
    "driest day":  [favourability(c, dry)[0] for c in classes],
}, index=classes).round(3)

Powdery mildew collapses on the wet day (free water bursts its spores) while
late blight peaks. Spider mites do the reverse. This is the structure that makes
the climate stream worth having.